# Smart Water Quality Assessment — Project Walkthrough

This notebook documents the execution workflow. The reusable preprocessing, training and evaluation code is maintained in `train.py` so the same steps can be repeated from the terminal.

## 1. Install dependencies
Run this cell in Google Colab or a local Jupyter environment if the dependencies are not installed.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib joblib streamlit

## 2. Download and inspect the dataset
The downloader saves the CSV under `data/water_potability.csv`. An internet connection is required for the first download.

In [ ]:
from pathlib import Path
import pandas as pd
from download_data import download_dataset

csv_path = download_dataset()
df = pd.read_csv(csv_path)
print("Shape:", df.shape)
display(df.head())
display(df.isna().sum().rename("missing values").to_frame())
display(df["Potability"].value_counts().sort_index().rename(index={0:"Not potable label (0)", 1:"Potable label (1)"}).to_frame("samples"))

## 3. Run preprocessing, EDA and model comparison
The script checks the required columns, removes duplicates, creates EDA plots, splits the data using stratification, and trains Logistic Regression, Decision Tree and Random Forest pipelines. Imputation is fitted inside each pipeline using the training partition only.

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, "train.py"], text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Training did not complete successfully. Review the error above.")

## 4. Review the experimental results

In [ ]:
metrics = pd.read_csv("results/model_comparison.csv")
display(metrics.round(3))
from IPython.display import Image, display as show_image
for filename in ["class_distribution.png", "missing_values.png", "correlation_heatmap.png"]:
    print(filename)
    show_image(Image(filename="results/" + filename))

## 5. Start the prototype
After training has created the saved model bundle, launch the Streamlit interface from a terminal:

```bash
streamlit run app.py
```

The interface displays dataset summaries, EDA, model comparison and a sample prediction form. Predictions reflect the benchmark dataset labels and are not a substitute for laboratory testing.